In [1]:
import os
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import LongType, StringType
from pyspark.sql.window import Window
import mitosheet

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

In [2]:
# --- ГЛОБАЛЬНЫЕ КОНФИГУРАЦИИ ПРИОРИТЕТОВ СИСТЕМНЫХ ПРЕФИКСОВ ---
PRIORITY_MAP = {
    "sync_rr_": 1, "sync_pr_": 2, "sync_chg_r_": 2, "sync_ca_": 3, "sync_cr_": 3, 
    "sync_init_c_": 3, "sync_ra_": 4, "sync_init_r_": 4, "sync_pa_": 5, 
    "sync_chg_a_": 5, "sync_init_p_": 6, "sync_epic_": 7
}
PRIORITY_EXPR = "CASE " + " ".join([f"WHEN message_id LIKE '{p}%' THEN {w}" for p, w in PRIORITY_MAP.items()]) + " ELSE 99 END"

In [3]:
def load_local_parquet():
    """Создает локальную Spark-сессию и загружает данные из папки DownloadedData."""
    spark = SparkSession.builder \
        .master("local[*]") \
        .appName("LocalReadApp") \
        .config("spark.driver.host", "localhost") \
        .config("spark.hadoop.fs.file.impl", "org.apache.hadoop.fs.RawLocalFileSystem") \
        .getOrCreate()

    path = "DownloadedData"
    if not os.path.exists(path):
        print(f"Ошибка: Путь '{path}' не найден.")
        return None
    try:
        df = spark.read.parquet(path)
        print(f"Успех! Загружено строк: {df.count()}")
        return df
    except Exception as e:
        print("Ошибка при чтении Parquet:", e)
        return None

In [4]:
def calculate_entity_lifecycles(df):
    """
    Рассчитывает время нахождения в статусе, дельты между сообщениями
    и размечает актуальные версии сущностей в стиле SCD2.
    Сортировка окон приведена к строгому системному стандарту.
    """
    # 1. Приведение даты к Timestamp и определение текущего момента
    df_prepared = df.withColumn(
        "ts_current", F.to_timestamp("entity_update_date", "yyyy-MM-dd'T'HH:mm:ss'Z'")
    ).withColumn(
        "ts_now", F.current_timestamp()
    )

    # 2. Определение детерминированных окон (согласованных с приоритетами генерации)
    win_lead = Window.partitionBy("entity_number").orderBy(
        "ts_current", "kafka_timestamp", F.asc("is_generated"), "sort_priority", "message_id"
    )
    
    win_status_desc = Window.partitionBy("entity_number", "entity_status_name").orderBy(
        F.desc("ts_current"), F.desc("kafka_timestamp"), F.desc("is_generated"), F.desc("sort_priority"), F.desc("message_id")
    )
    
    win_entity_desc = Window.partitionBy("entity_number").orderBy(
        F.desc("ts_current"), F.desc("kafka_timestamp"), F.desc("is_generated"), F.desc("sort_priority"), F.desc("message_id")
    )

    # 3. Расчет временных промежутков и флагов актуальности
    df_lifecycle = df_prepared.withColumn(
        "ts_next", F.lead("ts_current").over(win_lead)
    ).withColumn(
        "ts_end", F.coalesce(F.col("ts_next"), F.col("ts_now"))
    ).withColumn(
        "row_diff", F.col("ts_end").cast("long") - F.col("ts_current").cast("long")
    ).withColumn(
        "is_status_actual", F.when(F.row_number().over(win_status_desc) == 1, 1).otherwise(0)
    ).withColumn(
        "is_entity_actual", F.when(F.row_number().over(win_entity_desc) == 1, 1).otherwise(0)
    )

    # 4. Расчет кумулятивного времени
    win_cum = Window.partitionBy("entity_number", "entity_status_name").orderBy(
        "ts_current", "kafka_timestamp", F.asc("is_generated"), "sort_priority", "message_id"
    ).rowsBetween(Window.unboundedPreceding, Window.currentRow)
    
    win_total = Window.partitionBy("entity_number", "entity_status_name")

    df_final = df_lifecycle.withColumn(
        "cumulative_status_time", F.sum("row_diff").over(win_cum)
    ).withColumn(
        "total_status_time", F.sum("row_diff").over(win_total)
    )

    # Удаляем служебные колонки расчета времени, оставляя только ts_current и целевые метрики
    return df_final.drop("ts_now", "ts_next", "ts_end")

In [5]:
def prepare_global_attributes(df):
    """
    Шаг 0: Базовая подготовка датафрейма. 
    Извлекает спринты, дефолтит is_generated и рассчитывает сквозной sort_priority.
    """
    return df.withColumn(
        "sprint", F.expr("filter(sprints, x -> x.type = 'sprint' AND x.is_actual = 1)[0].name")
    ).withColumn(
        "sprint_key", F.expr("filter(sprints, x -> x.type = 'sprint' AND x.is_actual = 1)[0].sprint_key")
    ).withColumn(
        "supersprint", F.expr("filter(sprints, x -> x.type = 'supersprint' AND x.is_actual = 1)[0].name")
    ).withColumn(
        "supersprint_key", F.expr("filter(sprints, x -> x.type = 'supersprint' AND x.is_actual = 1)[0].sprint_key")
    ).withColumn(
        "is_generated", F.coalesce(F.col("is_generated").cast("boolean"), F.lit(False))
    ).withColumn(
        "sort_priority", F.expr(PRIORITY_EXPR))

In [6]:
def extract_epic_base(df):
    """Шаг 1: Парсит нативные RDS-связи Эпика. origin_feature = null."""
    return df.filter(F.col("entity_type_code") == "epic") \
        .withColumn("own_rds", F.expr("""
            transform(
                filter(relations, x -> x.relation_entity_type_code = 'rds'), 
                r -> struct(
                    r.relation_entity_number as rds_number, 
                    'direct' as relation_type, 
                    cast(null as string) as origin_feature
                )
            )
        """))

In [7]:
def generate_sync_epic_triggers(df):
    """Шаг 2: Генерирует триггеры из истории Фич. Вшивает entity_number Фичи как origin_feature."""
    features_base = df.filter(F.col("entity_type_code") == "feature") \
        .withColumn("rds_list", F.expr("array_distinct(transform(filter(relations, x -> x.relation_entity_type_code = 'rds'), r -> r.relation_entity_number))")) \
        .withColumn("parent_epic_list", F.expr("""
            array_distinct(concat(
                transform(filter(relations, x -> x.relation_entity_type_code = 'epic'), r -> r.relation_entity_number),
                case when entity_parent_number is not null and lower(entity_parent_number) != 'null' then array(entity_parent_number) else array() end
            ))
        """))
    
    win_feat_history = Window.partitionBy("entity_number").orderBy(
        "entity_update_date", "kafka_timestamp", F.asc("is_generated"), "sort_priority", "message_id"
    )

    features_with_lag = features_base \
        .withColumn("prev_epic_list", F.lag("parent_epic_list").over(win_feat_history)) \
        .withColumn("prev_rds_list", F.lag("rds_list").over(win_feat_history))

    features_exploded = features_with_lag.withColumn("epic_to_trigger", F.explode_outer("parent_epic_list"))

    return features_exploded.filter(
        F.col("epic_to_trigger").isNotNull() & 
        (F.size("rds_list") > 0) & 
        F.array_contains(F.coalesce(F.col("prev_epic_list"), F.array()), F.col("epic_to_trigger")) & 
        ((F.col("prev_rds_list").isNull()) | (F.col("rds_list") != F.col("prev_rds_list")))
    ).select(
        F.col("epic_to_trigger").alias("entity_number"),
        F.col("entity_update_date"),
        F.col("kafka_timestamp"),
        F.concat(F.lit("sync_epic_"), F.col("message_id")).alias("message_id"),
        F.lit("epic").alias("entity_type_code"),
        F.lit(True).alias("is_generated"),
        # ВНИМАНИЕ: кладем entity_number самой Фичи в поле origin_feature структуры
        F.expr("""
            transform(rds_list, x -> struct(
                x as rds_number, 
                'indirect' as relation_type, 
                entity_number as origin_feature
            ))
        """).alias("indirect_rds")
    ).distinct()

In [8]:
def apply_epic_snapshot_inheritance(epics_base, feature_triggers):
    """Шаг 3: Объединение Эпиков с триггерами и наследование текстовых атрибутов (Snapshot)."""
    epics_backbone = epics_base.unionByName(feature_triggers, allowMissingColumns=True) \
        .withColumn("sort_priority", F.expr(PRIORITY_EXPR))

    win_snapshot = Window.partitionBy("entity_number").orderBy(
        "entity_update_date", "kafka_timestamp", F.asc("is_generated"), "sort_priority", "message_id"
    )
    
    state_cols = [c for c in epics_backbone.columns if c not in [
        "entity_number", "entity_update_date", "kafka_timestamp", "sort_priority", 
        "message_id", "is_generated", "own_rds", "entity_type_code", "indirect_rds", "relations"
    ]]

    epics_backbone = epics_backbone.withColumn("prev_state", F.lag(F.struct(*[F.col(c) for c in state_cols])).over(win_snapshot))
    for c in state_cols:
        epics_backbone = epics_backbone.withColumn(c, F.coalesce(F.col(c), F.col(f"prev_state.{c}")))
        
    return epics_backbone.drop("prev_state"), win_snapshot

In [9]:
def merge_direct_and_indirect_rds(epics_backbone, win_snapshot, epics_raw_schema_names):
    """Шаг 4: Слияние структур. Изменен приоритет дедупликации: indirect теперь выигрывает у direct."""
    target_struct_type = "array<struct<rds_number:string, relation_type:string, origin_feature:string>>"
    
    epics_backbone = epics_backbone.withColumn("last_own_rds", F.last("own_rds", ignorenulls=True).over(win_snapshot))
    
    epics_backbone = epics_backbone.withColumn("all_structures", F.concat(
        F.coalesce("last_own_rds", F.array().cast(target_struct_type)),
        F.coalesce("indirect_rds", F.array().cast(target_struct_type))
    ))

    # Сортируем с кастомным лямбда-компаратором: 
    # Если rds_number одинаковые, то 'indirect' (i) должен идти ПЕРЕД 'direct' (d), поэтому сортируем по relation_type desc
    epics_backbone = epics_backbone.withColumn("total_rds", F.expr(f"""
        reduce(
            array_sort(all_structures, (left, right) -> 
                case 
                    when left.rds_number < right.rds_number then -1
                    when left.rds_number > right.rds_number then 1
                    when left.relation_type > right.relation_type then -1
                    when left.relation_type < right.relation_type then 1
                    else 0
                end
            ),
            cast(array() as {target_struct_type}),
            (acc, x) -> case when exists(acc, el -> el.rds_number = x.rds_number) then acc else array_union(acc, array(x)) end
        )
    """))

    epics_backbone = epics_backbone.withColumn("exploded_rds", F.explode_outer("total_rds"))

    # Извлекаем rds_number, тип связи и имя Фичи-первоисточника
    return epics_backbone \
        .withColumn("target_rds_number", F.col("exploded_rds.rds_number")) \
        .withColumn("linked_rds_relation_type", F.coalesce(F.col("exploded_rds.relation_type"), F.lit("undefined"))) \
        .withColumn("linked_rds_origin_feature", F.col("exploded_rds.origin_feature")) \
        .withColumn("is_main", F.lit(True)) \
        .select(epics_raw_schema_names + ["target_rds_number", "linked_rds_relation_type", "linked_rds_origin_feature", "is_main"])

In [10]:
def enrich_with_rds_attributes(df, epics_final, rds_columns):
    """Шаг 5: Сборка финального Union. Появилось сквозное поле linked_rds_origin_feature."""
    # Справочник RDS
    rds_flow = df.filter(F.col("entity_type_code") == "rds") \
        .withColumn("target_rds_number", F.col("entity_number")) \
        .withColumn("linked_rds_relation_type", F.lit("direct")) \
        .withColumn("linked_rds_origin_feature", F.lit(None).cast("string")) \
        .withColumn("is_main", F.lit(False))

    # Фичи
    features_final = df.filter(F.col("entity_type_code") == "feature") \
        .withColumn("rds_list", F.expr("array_distinct(transform(filter(relations, x -> x.relation_entity_type_code = 'rds'), r -> r.relation_entity_number))")) \
        .withColumn("target_rds_number", F.explode_outer("rds_list")) \
        .withColumn("linked_rds_relation_type", F.when(F.col("target_rds_number").isNotNull(), F.lit("direct")).otherwise(F.lit("undefined"))) \
        .withColumn("linked_rds_origin_feature", F.col("entity_number")) \
        .withColumn("is_main", F.lit(True)) \
        .select(df.schema.names + ["target_rds_number", "linked_rds_relation_type", "linked_rds_origin_feature", "is_main"])

    # Объединяем потоки. epics_final уже содержит linked_rds_origin_feature!
    final_combined = rds_flow.unionByName(epics_final, allowMissingColumns=True) \
                             .unionByName(features_final, allowMissingColumns=True)

    # Стандартный сквозной Forward Fill по атрибутам RDS
    win_final = Window.partitionBy("target_rds_number").orderBy(
        "entity_update_date", "kafka_timestamp", F.desc("is_generated"), "message_id", "is_main"
    )
    payload_expr = F.when(F.col("is_main") == False, F.struct(*[F.col(c) for c in rds_columns])).otherwise(F.lit(None))
    final_combined = final_combined.withColumn("last_rds_struct", F.last(payload_expr, ignorenulls=True).over(win_final))

    for col in rds_columns:
        res_col = f"linked_rds_{col}"
        if col == "entity_number":
            final_combined = final_combined.withColumn(res_col, F.coalesce(F.col(f"last_rds_struct.{col}"), F.col("target_rds_number")))
        elif col == "type_rds":
            final_combined = final_combined.withColumn(res_col, F.when(F.col("last_rds_struct").isNull() & F.col("target_rds_number").isNotNull(), F.lit("undefined")).otherwise(F.col(f"last_rds_struct.{col}")))
        else:
            final_combined = final_combined.withColumn(res_col, F.col(f"last_rds_struct.{col}"))

    return final_combined.drop("last_rds_struct", "is_main", "target_rds_number")

In [14]:
# --- ГЛАВНАЯ ФУНКЦИЯ-ОРКЕСТРАТОР ПАЙПЛАЙНА (КОНВЕЙЕР) ---
def enrich_with_rds_links(df, rds_columns=None):
    """
    Конвейерная функция, оркеструющая все этапы обогащения.
    Разделяет расчет жизненных циклов: Фичи и RDS считаются на старте, 
    Эпики — строго после слияния с триггерами (для честного времени sync_epic_).
    """
    if rds_columns is None:
        rds_columns = ["entity_number", "type_rds", "entity_status_name", "entity_priority_name", "entity_update_date"]

    # Шаг 0: Выделяем плоские спринты и генерируем глобальный sort_priority
    df_with_sprints = prepare_global_attributes(df)

    # Шаг 0.1: Считаем жизненный цикл для Фич и RDS на старте
    df_features_and_rds_lifecycle = calculate_entity_lifecycles(
        df_with_sprints.filter(F.col("entity_type_code").isin("feature", "rds"))
    )

    # Шаг 1: Готовим нативную базу Эпиков (чистый лог)
    epics_base = extract_epic_base(df_with_sprints)
    
    # Шаг 2: Генерируем точки синхронизации из истории Фич (на базе потока с таймингами)
    feature_triggers = generate_sync_epic_triggers(df_features_and_rds_lifecycle)
    
    # Шаг 3: Объединяем реальные Эпики и триггеры, наследуем паспорта (Snapshot)
    epics_backbone, win_snapshot = apply_epic_snapshot_inheritance(epics_base, feature_triggers)
    
    # Шаг 3.1: Считаем жизненный цикл для Эпиков (теперь sync_epic_ строки получат верное время!)
    epics_with_lifecycle = calculate_entity_lifecycles(epics_backbone)
    
    # Запоминаем расширенную схему полей Эпика (включая расчетные колонки lifecycles)
    extended_schema_names = [
        c for c in epics_with_lifecycle.columns 
        if c not in ["target_rds_number", "linked_rds_relation_type", "linked_rds_origin_feature", "is_main"]
    ]
    
    # Шаг 4: Слияние прямых/косвенных RDS и выполнение explode (с приоритетом INDIRECT > DIRECT)
    epics_final = merge_direct_and_indirect_rds(epics_with_lifecycle, win_snapshot, extended_schema_names)
    
    # Шаг 5: Общий Union и финальный Forward Fill атрибутов RDS
    result_df = enrich_with_rds_attributes(df_features_and_rds_lifecycle, epics_final, rds_columns)
    
    return result_df

In [16]:
# 1. Загружаем данные из Parquet
df = load_local_parquet()

# 1. Строим структуру, строго повторяющую схему из ошибки
# 1. Строим структуру, строго повторяющую схему из ошибки
new_rds_relation = F.struct(
    F.lit(None).cast(LongType()).alias("relation_id"),
    F.lit(None).cast(StringType()).alias("relation_type"),
    F.lit(None).cast(StringType()).alias("relation_properties"),
    F.lit("RDS-162243").alias("relation_entity_number"),
    F.lit(None).cast(StringType()).alias("relation_entity_name"),
    F.lit(None).cast(StringType()).alias("relation_entity_parent_number"),
    F.lit("rds").alias("relation_entity_type_code"),
    F.lit(None).cast(StringType()).alias("relation_entity_type_name"),
    F.lit(None).cast(StringType()).alias("relation_entity_status_code"),
    F.lit(None).cast(StringType()).alias("relation_entity_status_name"),
    F.lit(None).cast(StringType()).alias("relation_entity_priority_code"),
    F.lit(None).cast(StringType()).alias("relation_entity_priority_name")
)

# 2. Обновляем DataFrame
df = df.withColumn(
    "relations",
    F.when(
        F.col("message_id") == "e973eec8-ff56-42a9-8d95-f4331c97128d",
        F.array_union(
            F.coalesce(F.col("relations"), F.array()), 
            F.array(new_rds_relation)
        )
    ).otherwise(F.col("relations"))
)

# 2. Запускаем конвейер (он сам внутри сделает извлечение спринтов, 
#    расчет времени для всех сущностей, генерацию sync_epic и финальный union)
if df is not None:
    result_df = enrich_with_rds_links(df)
    
    # 3. Проверяем результат (выводим количество строк и схему, чтобы убедиться, что всё ок)
    print(f"Пайплайн успешно завершен! Итоговое количество строк: {result_df.count()}")
    # result_df.printSchema()
    
    # 4. Если нужно посмотреть на данные глазами через Pandas в Jupyter:
    # (берём первые 20 строк для примера, чтобы не перегружать память драйвера)
    # display(result_df.limit(20).toPandas())

Успех! Загружено строк: 586
Пайплайн успешно завершен! Итоговое количество строк: 604


In [ ]:
cols = ['message_id', 'action_type', 'entity_number', 'entity_update_date', 'entity_parent_number', 'entity_type_code', 'children', 'relations', 'type_rds', 'alert_id', 'is_generated', 
        'linked_rds_entity_number', 'linked_rds_relation_type', 'linked_rds_type_rds', 'linked_rds_entity_status_name', 'cumulative_status_time']

mitosheet.sheet(result_df.toPandas().loc[lambda _df: 
                ~(_df.linked_rds_entity_number.isna()) &
                ~(_df.action_type.isna()) &
                 (_df.linked_rds_type_rds == 'alert')
                ,cols].sort_values(by=['entity_update_date']))

In [17]:
test_df = load_local_parquet()

# 1. Строим структуру, строго повторяющую схему из ошибки
new_rds_relation = F.struct(
    F.lit(None).cast(LongType()).alias("relation_id"),
    F.lit(None).cast(StringType()).alias("relation_type"),
    F.lit(None).cast(StringType()).alias("relation_properties"),
    F.lit("RDS-162243").alias("relation_entity_number"),
    F.lit(None).cast(StringType()).alias("relation_entity_name"),
    F.lit(None).cast(StringType()).alias("relation_entity_parent_number"),
    F.lit("rds").alias("relation_entity_type_code"),
    F.lit(None).cast(StringType()).alias("relation_entity_type_name"),
    F.lit(None).cast(StringType()).alias("relation_entity_status_code"),
    F.lit(None).cast(StringType()).alias("relation_entity_status_name"),
    F.lit(None).cast(StringType()).alias("relation_entity_priority_code"),
    F.lit(None).cast(StringType()).alias("relation_entity_priority_name")
)

# 2. Обновляем DataFrame
test_df = test_df.withColumn(
    "relations",
    F.when(
        F.col("message_id") == "e973eec8-ff56-42a9-8d95-f4331c97128d",
        F.array_union(
            F.coalesce(F.col("relations"), F.array()), 
            F.array(new_rds_relation)
        )
    ).otherwise(F.col("relations"))
)

# 2. Запускаем конвейер (он сам внутри сделает извлечение спринтов, 
#    расчет времени для всех сущностей, генерацию sync_epic и финальный union)
if test_df is not None:
    test_result_df = enrich_with_rds_links(test_df)
    
    # 3. Проверяем результат (выводим количество строк и схему, чтобы убедиться, что всё ок)
    print(f"Пайплайн успешно завершен! Итоговое количество строк: {test_result_df.count()}")
    # result_df.printSchema()
    
    # 4. Если нужно посмотреть на данные глазами через Pandas в Jupyter:
    # (берём первые 20 строк для примера, чтобы не перегружать память драйвера)
    # display(result_df.limit(20).toPandas())

Успех! Загружено строк: 586
Пайплайн успешно завершен! Итоговое количество строк: 604


In [ ]:
# mitosheet.sheet(test_df.toPandas().loc[lambda _df: _df.entity_number == 'DEMOSTREAM-734'].sort_values(by=['entity_update_date']), analysis_to_replay="id-bezyvnzzrl")

In [ ]:
cols = ['message_id', 'action_type', 'entity_number', 'entity_update_date', 'entity_parent_number', 'entity_type_code', 'children', 'relations', 'type_rds', 'alert_id', 'is_generated', 
        'linked_rds_entity_number', 'linked_rds_relation_type', 'linked_rds_type_rds', 'linked_rds_entity_status_name', 'cumulative_status_time', 'is_status_actual', 'is_entity_actual']

mitosheet.sheet(test_result_df.toPandas().loc[lambda _df: 
                ~(_df.linked_rds_entity_number.isna()) &
                ~(_df.action_type.isna()) &
                 (_df.linked_rds_type_rds == 'alert')
                ,cols].sort_values(by=['entity_update_date', 'is_generated'], ascending=[False, False]))